# How do you know your agent works?
### Evaluating Earth Observation agents: benchmarks, metrics, and a reusable harness

Hands-on session, BIFOLD Agentic AI for Earth Observation workshop. Pi School and ESA Φ-lab, EVE project.

Building an agent is straightforward. Showing that it works reliably is harder.

### What changes compared with evaluating an LLM

Scoring a plain LLM is mostly one question: *is the answer right?* The model gets a prompt and returns text, so you compare that text with a reference (accuracy, F1, BLEU, a judge). An **agent** is different:

- **It acts, not only answers.** It plans, calls tools (raster readers, index calculators, GIS operations, map APIs, code), reads the results, and decides what to do next. The answer is the last step of a longer process.
- **A right answer can hide a bad process.** The agent can guess, take a lucky shortcut, or waste ten calls. A wrong answer can come from one bad step in an otherwise sound run. The final answer alone cannot tell these apart.
- **The path is part of the result.** Which tools it chose, in which order, with which arguments, how many calls, and how it recovered from errors are all measurable, but only if the run is recorded as an **execution trace**.
- **Runs are not repeatable by default.** The same agent can take a different path on the same question, so you need fixed tasks, fixed tools, recorded configurations, and often several runs.
- **The environment is part of the system.** Tool schemas, data, the model endpoint, and timeouts change the score, so a number is only meaningful with the setup that produced it.

That is why agent evaluation has two kinds of metrics: **end-to-end** metrics (did it get the result?) and **step-by-step** metrics (did it get there the right way?).

### How this notebook is organised

- **Part 1. Metrics for agents**
- **Part 2. EO benchmarks for agents**
- **Part 3. The evaluation framework, and a real Earth-Bench run**

The model under test is EVE, served through an OpenAI-compatible endpoint. Any other endpoint works the same way.

## Setup

Select the kernel **agentic-evals workshop**.

In [1]:
from pathlib import Path
import os, sys

root = Path.cwd()
if not (root / "workshop.py").is_file():
    candidate = root / "evaluating-eo-agents-bifold"
    if (candidate / "workshop.py").is_file():
        root = candidate
    else:
        raise RuntimeError("Run this notebook with the working directory set to the evaluating-eo-agents-bifold folder.")
os.chdir(root)
sys.path.insert(0, str(root))
sys.modules.pop("workshop", None)   # re-running this cell picks up edits to workshop.py

from dotenv import load_dotenv
load_dotenv(root / ".env")
if not os.environ.get("EVE_BASE_URL", "").strip() and os.environ.get("EVE_API_BASE_URL", "").strip():
    os.environ["EVE_BASE_URL"] = os.environ["EVE_API_BASE_URL"].strip()

os.environ["AGENTIC_EVALS_DATA"] = str((root / "data").resolve())
os.environ["AGENTIC_EVALS_RUNS_DIR"] = str((root / "runs").resolve())
# Tool servers write their banners, warnings and tracebacks to runs/.../kit_stderr.log, not here.
os.environ["AGENTIC_EVALS_KIT_STDERR"] = "file"
# With W&B off, turn the inspect-wandb hooks off too. Otherwise they log a warning on every event.
if os.environ.get("WANDB_MODE", "").strip() == "disabled":
    os.environ["INSPECT_WANDB_MODELS_ENABLED"] = "false"
    os.environ["INSPECT_WANDB_WEAVE_ENABLED"] = "false"

assert sys.version_info >= (3, 12), f"Python 3.12+ needed, found {sys.version.split()[0]}"
if "agentic_evals.paths" in sys.modules:
    from agentic_evals.paths import DATA_DIR
    expected = (root / "data").resolve()
    if DATA_DIR != expected:
        raise RuntimeError("Restart the kernel so AGENTIC_EVALS_DATA is picked up.")

print("folder", root)
print("python", sys.executable)

folder /Users/antoniolopez/pischool/agentic-evals/evaluating-eo-agents-bifold
python /Users/antoniolopez/pischool/agentic-evals/.venv1/bin/python


## Install the library

The lab already provides this environment, so the install cell below is commented out. If you run the notebook somewhere else, uncomment it. It installs `agentic-evals` from GitHub into this kernel, which needs `uv` on the path. Restart the kernel afterwards if `agentic_evals` was already imported.

In [2]:
# Install agentic-evals into this kernel. Commented out: the lab already provides the environment.
# Uncomment these lines if you run the notebook somewhere else.
# import subprocess, sys
# subprocess.run([
#     "uv", "pip", "install", "--python", sys.executable,
#     "agentic-evals[earth-bench] @ git+https://github.com/eve-esa/agents-evals.git@feat/workshop-notebook",
# ], check=True)


In [3]:
import pandas as pd
from workshop import (
    WORKSHOP_QUESTIONS, check_environment, download_earth_bench, judge, results_df, run,
)
from agentic_evals.agents.langgraph_react import langgraph_agent

pd.set_option("display.precision", 2)
MODEL_NAME = os.environ["MODEL_NAME"].strip()
MODEL = f"openai-api/eve/{MODEL_NAME}"
print("model under test", MODEL)
print("workshop questions", list(WORKSHOP_QUESTIONS))
check_environment()

model under test openai-api/eve/jsc/alias-eve
workshop questions [11, 2, 7, 8, 19, 21, 26, 31, 36, 42]
python 3.13.2
executable /Users/antoniolopez/pischool/agentic-evals/.venv1/bin/python
data /Users/antoniolopez/pischool/agentic-evals/evaluating-eo-agents-bifold/data/earth_bench
runs /Users/antoniolopez/pischool/agentic-evals/evaluating-eo-agents-bifold/runs
tool kits Analysis, Index, Inversion, Perception, Statistics
perception lookup True
rasters on disk for 10/10 workshop questions
endpoint identified True 
token budget available


## Part 1. Metrics for agents

Agent metrics come in two kinds, named after what they read: **step-by-step** metrics read the trajectory, and **end-to-end** metrics read the finished run. Earth-Bench and ThinkGeo both name their metrics this way.

### 1.1 Step-by-step metrics

Step-by-step metrics score the **process**: which tools the agent called, in which order, with which arguments, and how many calls it needed.

#### What is a trajectory?

A **trajectory** is the ordered list of tool calls an agent made: for each call, the tool's name and its arguments, and (in some benchmarks) the observation the tool returned. A benchmark that scores trajectories ships a **reference trajectory** written and checked by experts, and compares the agent's list with it. A real example, **Earth-Bench question 11**, shown as the expert's three calls in order:

![The gold trajectory of Earth-Bench question 11: get_filelist, lst_multi_channel, calculate_threshold_ratio](figures/trajectory_q11.png)

Other benchmarks record the same thing with small differences: a GeoBenchX task can have several reference solutions, and for an unsolvable task the reference is the single step `reject_task()` (refusing is the correct behaviour). EO-Gym also stores the **observation** returned by each step.

The point of recording this: two agents can give the same final answer through very different trajectories, and an agent can follow the expert path and still answer wrong. Only the trajectory shows which.

#### The metrics

Let $t^*$ be the expert tool list with $m$ steps and $t$ the agent's list. Earth-Bench ([Feng et al., ICLR 2026](https://arxiv.org/abs/2509.23141), Appendix B.2) defines them like this, and the other benchmarks use close variants:

- **Tool-Any-Order (TAO).** The share of expert tools that appear anywhere in the agent's list, $\dfrac{\lvert \mathrm{Set}(t^*) \cap \mathrm{Set}(t) \rvert}{\lvert \mathrm{Set}(t^*) \rvert}$. Did the agent find the right tools?
- **Tool-In-Order (TIO).** Walk through the expert tools in order and look for each one later in the agent's list than the previous match. Count matches until one is missing, divide by $m$. Extra calls in between are allowed. Did it use them in the right order?
- **Tool-Exact-Match (TEM).** The longest common prefix of $t$ and $t^*$, divided by $m$. No extra calls allowed. Did it follow the expert path with no detours?
- **Parameter Accuracy (ParamAcc).** Like TEM, but each call must also have the expert's arguments. Did it call them correctly? The next part explains how.
- **Step accuracy** (ThinkGeo). Instruction, tool, argument and summary correctness judged for each step of the run.

Five runs of the same three-step task, scored with the harness's own functions. Pick a run in the dropdown to see its trace next to the expert's:

In [4]:
def call(name, **kw):
    return {"name": name, "input": kw}

gold = [
    call("get_filelist", dir_path="benchmark/data/question11"),
    call("lst_multi_channel", band31_path="b31.tif", band32_path="b32.tif", output_path="lst.tif"),
    call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=310.0),
]
runs = {
    "A) The expert path":              gold,
    "B) An extra call first":          [gold[0], gold[0], gold[1], gold[2]],
    "C) Right tools, wrong threshold": [gold[0], gold[1], call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=300.0)],
    "D) Skips get_filelist":           [gold[1], gold[2]],
    "E) A different tool in step 2":   [gold[0], call("band_ratio", band_a="b31.tif", band_b="b32.tif"), gold[2]],
}
from workshop import compare_widget

compare_widget(runs, gold)

In the trace view, green rows match the expert, amber rows use the right tool with different arguments, red rows are a different or missing tool, and grey rows are extra calls. Wavy underlines mark the arguments that differ. Pick each run in the dropdown and read its scores:

- **B** wastes one call. TAO and TIO forgive it; TEM and ParamAcc drop to 1/3 because every later step is shifted by one.
- **C** uses the right tools in the right order, so TAO, TIO and TEM are all 1. Only ParamAcc sees the wrong threshold (2/3).
- **D** never lists the files. TIO is 0 because it looks for the first expert tool first and stops when it is missing.
- **E** picked another tool in step 2, which is the case the next part is about.

#### How parameter accuracy is computed

If the agent called a different tool, there are no arguments to compare with. **The tool name is checked first**, and the arguments only matter once the tool matches. The benchmarks line the calls up differently:

- **Earth-Bench (and our GeoBenchX step metrics): position by position**
  - Call *i* of the agent is compared with step *i* of the expert.
  - A step counts only if the tool name matches **and** the arguments are exactly equal: every key and value (file paths, thresholds, band order).
  - Scoring stops at the first step that fails either test, so later steps do not count even if they would match.
  - ParamAcc is the number of correct steps divided by $m$, so it can never be higher than TEM.
  - A wrong tool at step *i* makes step *i* wrong, because there are no arguments to compare.
- **EO-Gym: align by tool name, then compare**
  - For each expert step, scan forward for the first agent call with the **same tool name**. If there is none, the step scores 0 and no arguments are compared.
  - If there is one, its arguments are compared after normalisation: keys lowercased and sorted, whitespace collapsed, paths lowercased with slashes and cache ids normalised, numbers equal within 1e-6.
  - Its returned observation must match too. Both must match for the step to count.
  - Extra calls and calls to other tools are skipped over, not punished.

Run E under both rules. Its later calls are also spelled differently from the expert's (a path in another case, a cache id, a float changed at the 7th decimal), to show EO-Gym's normalisation:

In [5]:
from IPython.display import HTML, display
from agentic_evals.tasks.eo_gym import metrics as eo_gym
from workshop import rules_html

# Run E, written as EO-Gym steps (name, arguments, observation). The agent calls band_ratio in step 2,
# and spells the later paths, cache id and float slightly differently from the expert.
expected = [
    eo_gym.Step("get_filelist", {"dir_path": "d"}, '["a.tif"]'),
    eo_gym.Step("lst_multi_channel", {"band31_path": "A.tif", "output_path": "o/lst_ab12cd34ef56.tif"}, "saved"),
    eo_gym.Step("calculate_threshold_ratio", {"threshold": 310.0}, "49.7"),
]
actual = [
    eo_gym.Step("get_filelist", {"dir_path": "d"}, '["a.tif"]'),
    eo_gym.Step("band_ratio", {"x": 1}, "0.3"),
    eo_gym.Step("lst_multi_channel", {"band31_path": "a.tif", "output_path": "o/LST_00000000aaaa.tif"}, "saved"),
    eo_gym.Step("calculate_threshold_ratio", {"threshold": 310.0000001}, "49.7"),
]
display(HTML(rules_html(expected, actual)))

Same agent behaviour, different outcome: Earth-Bench's ParamAcc is 0.33 because scoring stops at the wrong tool in step 2, and EO-Gym's parameters score is 1.0 because it skips the detour and normalises spelling. Both are strict in a way you should know about: an equally valid call with a different output file name is a mismatch. That is one reason benchmarks also use judges.

#### Efficiency, errors and termination

- **Efficiency / length.** Earth-Bench reports $\lvert t \rvert / \lvert t^* \rvert$, a ratio (above 1 means more calls than the expert). GeoBenchX reports a signed relative difference, $(n - \bar{m}) / \bar{m}$.
- **Error rate.** The share of tool calls that fail (EO-Gym's function-call error rate; our `tool_errors` count) or of generated code that does not run (UnivEARTH).
- **Termination.** Did the run finish, or hit a step or model-call limit? This harness records it on every sample.

One real run, with each of these numbers:

![Cost and reliability of one Earth-Bench run: efficiency, tool errors, termination](figures/run_anatomy.png)

### 1.2 End-to-end metrics

End-to-end metrics score the **finished run**, without looking at how it got there: is the answer right, how often across attempts, and did the run end normally (termination, above).

#### Accuracy / exact match

The extracted answer equals the gold one. Earth-Bench reads the letter from `<Answer>C<Answer>` and compares it with the gold option. Cheap and objective, but a right answer says nothing about the path, and four options mean a blind guess is right 25% of the time.

#### Pass@k

An agent is stochastic, so one run is one sample of its behaviour. **Pass@k is the probability that at least one of k attempts at a task is correct**, averaged over tasks. Pass@1 is the reliability of a single try; a large gap between Pass@1 and Pass@k means the agent *can* solve the task but not *consistently*.

Think of all possible runs on one task as a **tree of traces**: at each step the agent picks a tool, reads the result and branches again, and each path ends in a right or a wrong answer. The numbers on the edges are the **probability of taking that branch**. Multiply them along a path to get the probability of that trace, and add the green leaves to get $p$, the chance that one attempt is correct. With $k$ independent attempts, the expected Pass@k is $1 - (1-p)^k$ (Chen et al., 2021, Appendix A). Multiplying along a path is the standard probability of a trajectory under a stochastic policy (OpenAI Spinning Up, *Key Concepts in RL*, section *The RL Problem*); the tree and its edge numbers are illustrative.

![Tree of possible traces for question 11, with three sampled attempts and the resulting Pass@1, Pass@2 and Pass@3](figures/pass_at_k_tree.png)

You cannot read the edge numbers off a real agent, so you estimate: run $n$ attempts per task, count the $c$ correct ones, and use the unbiased estimator of Chen et al. (2021, §2.1, Eq. 1):

$$\text{Pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

For the three attempts drawn in the figure, $n = 3$ and $c = 2$, so Pass@1 $= 1 - \binom{1}{1}/\binom{3}{1} = 0.67$ and Pass@2 = Pass@3 = 1. Do not plug the observed rate $\hat p = c/n$ into $1 - (1-\hat p)^k$: Chen et al. show that it underestimates Pass@k (Appendix A, Figure 13). The metric itself goes back to Kulal et al. (2019), who generate k samples per problem and count the problem solved if any sample passes (as Chen et al. describe it in §2.1).

Pass@k costs k times the tokens of one run. In this harness each attempt is one Inspect *epoch*.

#### LLM-as-a-judge

Exact match fails on free-text answers, and on tasks with several valid solution paths. A **judge** is a language model that compares a candidate answer with the gold answer and returns a verdict. Benchmarks use it for the official answer score:

- **GeoBenchX.** The judge compares the whole solution with the reference solutions. Its answer is a score: **0** no match, **1** partial, **2** match.
- **EO-Gym.** The judge decides whether the candidate and the reference **have the same meaning**. Its answer is **true** or **false**.
- **ThinkGeo.** The judge checks each prediction against curated questions written for the query. Its answer is **correct** or **incorrect**.

> **The judge's answer must follow the format the harness reads.**
>
> EO-Gym: `{"is_same_meaning": true, "reason": "49.74% is the value of option C"}`
>
> GeoBenchX: `<MATCH SCORE>1</MATCH SCORE>`

The judge model is part of the number, so always report it. A reply without the key or the tag is left unscored, not guessed. This is how the harness reads GeoBenchX's reply:

In [6]:
from agentic_evals.tasks.geobenchx import metrics as geobenchx

reply = (
    "<MATCH REASONING>The candidate loads the same dataset and filters the same countries, "
    "but colours the map by a different column.</MATCH REASONING>\n<MATCH SCORE>1</MATCH SCORE>"
)
print(geobenchx.parse_verdict(reply))                  # 0 no match, 1 partial, 2 match
print(geobenchx.parse_verdict("Looks right to me!"))   # no tag: the sample is left unscored, not guessed

(1, 'The candidate loads the same dataset and filters the same countries, but colours the map by a different column.')
(None, 'judge output has no <MATCH SCORE> tag')


Below, one candidate answer is scored by two judges against the gold answer. The candidate is free text. The gold is option C of question 11 (49.74%). Both judges get the same prompt, EO-Gym's semantic-equivalence prompt, and each returns one score: **1** same meaning, **0** different meaning.

The two judges are GPT-4.1 mini and Mistral Small 3.2, both on OpenRouter (`OPENROUTER_API_KEY`). They are set in `JUDGES` in `workshop.py`, which is where the EVE models go once the EVE budget is available.

In [7]:
import re

from IPython.display import HTML, display

from agentic_evals.tasks.earth_bench.dataset import load_samples
from workshop import JUDGES, judge_answer, judges_html

ap11 = next(s for s in load_samples("ap", "spectrum") if s.id == 11)
question = ap11.input.split("Question:", 1)[1]
stem = re.split(r"\nA\.", question)[0].replace("benchmark/data/question11", "").strip()
gold = "C. " + re.search(r"^C\.(.+)$", question, re.M).group(1).strip()
candidate = "About 50 percent of the area is above 310 K, so the answer is C."

verdicts = [await judge_answer(model, stem, gold, candidate, base_url=url) for model, url in JUDGES]
display(HTML(judges_html(stem, gold, candidate, verdicts)))

inspect_ai v0.3.280
- hooks enabled: 3
  agentic_evals/langfuse_scores: Attach each sample's scores to its Langfuse trace.
  agentic_evals/uncapped_max_tokens: Treat --max-tokens -1 as no output cap.
  agentic_evals/wandb_compare: Log shared summary metrics and trace tables onto the W&B run.

┌────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────┐
│ /Users/antoniolopez/pischool/agentic-evals/.venv1/lib/python3.13/site-packages/inspect_ai/model/_model.py:1623 │
│ in generate                                                                                                    │
│                                                                                                                │
│ /Users/antoniolopez/pischool/agentic-evals/.venv1/lib/python3.13/site-packages/inspect_ai/model/_providers/ope │
│ nai_compatible.py:301 in generate                                                                              │
│                                                                                                                │
│ /Users/antoniolopez/pischool/agentic-evals/.venv1/lib/python3.13/site-packages/inspect_ai/model/_providers/ope │
│ nai_compatible.py:384 in _generate_completion                                                                  │
│                                                                                                                │
│ /Users/antoniolopez/pischool/agentic-evals/.venv1/lib/python3.13/site-packages/openai/resources/chat/completio │
│ ns/completions.py:2952 in create                                                                               │
│                                                                                                                │
│   2949 │   │   timeout: float | httpx2.Timeout | None | NotGiven = not_given,                                  │
│   2950 │   ) -> ChatCompletion | AsyncStream[ChatCompletionChunk]:                                             │
│   2951 │   │   validate_response_format(response_format)                                                       │
│ > 2952 │   │   return await self._post(                                                                        │
│   2953 │   │   │   "/chat/completions",                                                                        │
│   2954 │   │   │   body=await async_maybe_transform(                                                           │
│   2955 │   │   │   │   {                                                                                       │
│                                                                                                                │
│ /Users/antoniolopez/pischool/agentic-evals/.venv1/lib/python3.13/site-packages/openai/_base_client.py:2053 in  │
│ post                                                                                                           │
│                                                                                                                │
│   2050 │   │   opts = FinalRequestOptions.construct(                                                           │
│   2051 │   │   │   method="post", url=path, json_data=body, content=content, files=await async_                │
│   2052 │   │   )                                                                                               │
│ > 2053 │   │   return await self.request(cast_to, opts, stream=stream, stream_cls=stream_cls)                  │
│   2054 │                                                                                                       │
│   2055 │   async def patch(                                                                                    │
│   2056 │   │   self,                                                                                           │
│                                                                                                                │
│ /Users/antoniolopez/pischool/agentic-evals/.venv1/lib/python3.13/site-packages/openai/_base_client.py:1838 in  │
│ request                                                                                                        │
│                                                                                         

What to remember about judges:

- **The verdict depends on the judge model.** Scores from different judges are not comparable, so the harness records the judge with every verdict and keeps each judge as its own result.
- **Judges can disagree with the metrics and with each other.** `agentic_evals.scorers.agreement` reports raw agreement and Cohen's kappa between any two verdicts stored in a log.
- **They cost a model call per sample** and can favour answers that look like their own style.

## Part 2. EO benchmarks for agents

A benchmark is a set of tasks with gold answers (and sometimes gold trajectories) and a published scoring protocol. This part surveys the benchmarks for agents on Earth Observation: what each one asks the agent to do, what it ships as gold, how it is scored, and whether you can run it.

### 2.1 Overview

**Gold** is the reference a benchmark ships: a gold *trajectory* of tool calls (step metrics are possible), a gold *plan* that is never executed, or only the final *answer*. Sizes, gold type and data availability come from our literature review (July 2026); a dash means the review does not record it, so check the paper.

| Benchmark | Focus | Size | Gold | Metrics | Data |
| --- | --- | --- | --- | --- | --- |
| **[Earth-Bench](https://arxiv.org/abs/2509.23141)** (Feng et al., ICLR 2026) | Raster analysis on spectral, product and RGB data, 104 tools | 248 tasks | Trajectory | Accuracy; TAO, TIO, TEM, ParamAcc, efficiency | HF |
| **[GeoBenchX](https://arxiv.org/abs/2503.18129)** (Krechetova & Kochedykov, SIGSPATIAL 2025) | GIS workflows on statistical, vector and raster data, including tasks that must be rejected | 202 tasks | Trajectory | LLM-judge match / partial / no match; length difference | GitHub |
| **[EO-Gym](https://arxiv.org/abs/2605.01250)** | Multimodal tasks with imagery in the prompt, local tool server | 7.64K train, 1.44K test | Trajectory | Pass@k with LLM judge; binary tool metrics; parameters | HF |
| **[ThinkGeo](https://arxiv.org/abs/2505.23752)** (Shabbir et al., CVPR 2026) | Optical and SAR remote sensing, 14 tools | 486 tasks, 1,778 steps | Trajectory | Judge-based answer accuracy; instruction, tool, argument, summary accuracy | HF |
| **[OpenEarthAgent](https://huggingface.co/datasets/MBZUAI/OpenEarthAgent)** (MBZUAI) | Geospatial queries with multimodal inputs and tool calls | 800 train, 400 test | Trajectory | – | HF |
| **[GeoAgentBench](https://arxiv.org/abs/2604.13888)** | Local GIS/EO toolbox over MCP, 6 domains | 57 tasks | Trajectory (3–17 calls) | TAO, TIO, TEM, PEA; VLM judge for maps | GitHub |
| **[TerraLogic](https://arxiv.org/abs/2607.12497)** | Optical, SAR and IR: perception, then spatial relations, then statistics | 545 tasks | Trajectory | – | HF |
| **[TerraBench](https://arxiv.org/abs/2606.13148)** | Earth-science reasoning, 3 tracks, 8 domains | 403 tasks, 24,500 steps | Trajectory | Process-level tool-use metrics; tolerance-aware numeric scoring | gated |
| **[DORA](https://arxiv.org/abs/2605.11633)** | Disaster response over a 108-tool MCP library | 515 tasks, 3,500 steps | Trajectory | – | planned |
| **[GeoNatureAgent](https://github.com/gabrielireland/GeoNatureAgent_Benchmark)** | Environmental analysis against a production API, 16 tools | 93 tasks | Expected tools | Expected tools and actions; required and forbidden strings; round and cost limits | GitHub |
| **[ShapefileGPT](https://arxiv.org/abs/2410.12376)** | Shapefile processing with a 27-function API | 42 tasks | Trajectory | Accuracy; success rate; parameter and repeat-call metrics | no |
| **[GeoJSON Agents](https://arxiv.org/abs/2509.08863)** | GeoJSON analysis: function calling vs code generation | 70 tasks | Trajectory | – | Figshare |
| **[GeoLLM-Engine](https://arxiv.org/abs/2404.15500)** | Geospatial copilot in a web map UI, >175 tools | 100,000 queries | Trajectory and final state | Model checker on tool sequence, reply and final state | no |
| **[GeoLLM-QA](https://arxiv.org/abs/2405.00709)** | UI-grounded copilot tasks on live map state | ~1,000 tasks | Trajectory | – | no |
| **[RescueADI](https://arxiv.org/abs/2410.13384)** | Disaster-scene interpretation with perception tools | 13,424 requests | Plan and answer | – | no |
| **[KnowFlow-Bench](https://arxiv.org/abs/2512.15231)** | Remote-sensing workflows as DAGs of GDAL and deep-learning tools | 324 workflows | Plan | – | no |
| **[GeoPlan-bench](https://arxiv.org/abs/2511.17198)** | Remote-sensing planning, 7 sub-domains, tools not executed | 1,244 tasks | Plan | Key-tool recall, precision, F1; path similarity; Elo completeness (LLM judge) | GitHub |
| **[DisasterBench](https://arxiv.org/abs/2605.27957)** | Disaster planning over 26 typed agent manifests | 233 tasks | Plan | Exact structured-plan match | GitHub |
| **[UnivEARTH](https://arxiv.org/abs/2504.12110)** (Kao et al., 2025) | Yes/no questions solved with Earth Engine code | 140 questions | Answer | Accuracy; code failure rate | HF |
| **[GeoCode-Bench](https://arxiv.org/abs/2410.09738)** (Hou et al., 2024) | Geospatial code generation | – | Answer | Answer accuracy | – |
| **[GeoCode](https://arxiv.org/abs/2410.18792)** (GeoAgent) | Earth Engine and 28-library code, single and multi-turn | 2 splits | Answer (gold code) | pass@1; task completion rate | GitHub |
| **[MapEval-API](https://arxiv.org/abs/2501.00316)** (Dihan et al., ICML 2025) | Map-API questions: places, routes, trips | 300 questions | Answer | Accuracy | HF |
| **[OpenEarth-Bench](https://arxiv.org/abs/2603.22148)** | Full EO pipelines where the agent may create tools, 7 domains | 596 cases | Answer | Stage-wise and end-to-end accuracy; debug rounds; running time | no |
| **[GeoDisaster](https://arxiv.org/abs/2606.17246)** | Disaster reasoning on optical, SAR, vectors and road networks | 2,921 instances | Answer | – | no |
| **[MineBench](https://arxiv.org/abs/2412.17339)** (MineAgent) | Mineral exploration as multi-image binary VQA | 612 areas | Answer | – | no |
| **[VagueEO](https://github.com/1e12Leon/RemoteAgent)** (RemoteAgent) | Vague non-expert queries routed to expert models over MCP | 5,000 train, 10 test tasks | Answer (labels, boxes, masks) | – | no |
| **[AirSpatial-Bench](https://arxiv.org/abs/2601.01416)** | Drone imagery: vehicle attributes and retrieval in 3D | 1,773 questions | Answer | – | HF |
| **[AgroTools](https://arxiv.org/abs/2605.22366)** | Agricultural remote-sensing VQA, about half SPOT 6-7 | 539 samples | Answer | – | HF, without tools or grader |
| **[AgriWorld / AgroBench](https://arxiv.org/abs/2602.15325)** | Agricultural science with code execution | ~541 tasks | Answer | – | no |
| **[ZephyrusBench](https://arxiv.org/abs/2510.04017)** (ICLR 2026) | Weather science with code on WeatherBench-2 | 2,230 questions | Answer | – | HF |
| **[AgentCaster](https://arxiv.org/abs/2510.03349)** | Tornado forecasting on an HRRR archive | 40 forecast days | Answer (risk polygons) | TornadoBench; TornadoHallucination | HF |
| **[Climate-Agent-Bench-85](https://arxiv.org/abs/2511.20109)** (ClimateAgent) | Climate data-science workflows with a multi-agent system | 85 tasks | Answer | Task completion; report quality | no |
| **[GCA-DS](https://arxiv.org/abs/2604.12306)** (ACL 2026) | Gulf climate decision support, multimodal QA | ~200k QA, ~91 agentic tasks | Answer | – | no |

Three things to take from this table:

1. **Every benchmark brings its own metrics.** Compare numbers within one benchmark, not across them.
2. **The same name can mean different things.** "Tool exact match" is a fraction in Earth-Bench and a 0/1 flag in EO-Gym (section 1.1).
3. **LLM judges are part of some protocols.** For GeoBenchX, EO-Gym and ThinkGeo the official answer score comes from a judge model, so the judge must be reported with the number.


### 2.2 Many ways to group the same benchmarks

Each axis below splits the table differently, which is why no single ranking of "the best EO agent" exists. Pick the axes that match your agent first, then the benchmarks.

- **Gold: trajectory, plan or answer.** Gold trajectories allow step metrics: Earth-Bench, GeoBenchX, EO-Gym, ThinkGeo, GeoAgentBench, TerraBench, DORA. Gold plans are never executed: GeoPlan-bench, DisasterBench, KnowFlow-Bench. Answer only: UnivEARTH, MapEval-API, ZephyrusBench, AgentCaster, GeoDisaster.
- **Does the model need vision?** Imagery in the prompt needs a vision-language model: EO-Gym, MineBench, VagueEO, AirSpatial-Bench, AgroTools. Imagery reached only through tools works with a text-only model: Earth-Bench, ThinkGeo, TerraLogic. No imagery at all: GeoBenchX, MapEval-API, ShapefileGPT, GeoJSON Agents, ZephyrusBench.
- **Data modality.** Optical RGB (EO-Gym, ThinkGeo, AirSpatial-Bench), SAR (ThinkGeo, TerraLogic, GeoDisaster, DORA), multispectral and hyperspectral (Earth-Bench, MineBench, DORA), infrared (TerraLogic), vector GIS (GeoBenchX, ShapefileGPT, GeoJSON Agents), gridded weather (ZephyrusBench, AgentCaster), map services (MapEval-API).
- **How the agent acts.** A fixed tool set: Earth-Bench (104 tools), DORA (108), GeoBenchX (24), ShapefileGPT (27), ThinkGeo (14). Writing and running code: UnivEARTH (Earth Engine), GeoCode, ZephyrusBench, AgriWorld, ClimateAgent; OpenEarth-Bench also lets the agent create tools. Clicks and map state in a web UI: GeoLLM-Engine, GeoLLM-QA. Planning only, with no execution: GeoPlan-bench, DisasterBench.
- **Answer format.** Multiple choice (Earth-Bench, MapEval-API), yes/no or binary (UnivEARTH, MineBench), numbers within a tolerance (TerraBench), boxes and masks (VagueEO, AirSpatial-Bench), maps and files (GeoBenchX, GeoAgentBench), polygons (AgentCaster), DAG plans (DisasterBench, KnowFlow-Bench), reports (ClimateAgent).
- **Who scores.** A deterministic match (Earth-Bench, MapEval-API, DisasterBench), an execution or state check (GeoCode pass@1, the GeoLLM-Engine model checker), or an LLM or VLM judge inside the protocol (GeoBenchX, EO-Gym, ThinkGeo, GeoPlan-bench, GeoAgentBench maps).
- **Must the agent know when to stop?** GeoBenchX has unsolvable tasks that must be rejected, EO-Gym accepts "next frame not available" as an answer, and AgentCaster scores hallucinated risk separately.
- **Application domain.** General remote sensing (Earth-Bench, EO-Gym, ThinkGeo, TerraLogic), disasters (RescueADI, DisasterBench, DORA, GeoDisaster), weather and climate (ZephyrusBench, AgentCaster, ClimateAgent, GCA-DS), agriculture (AgroTools, AgriWorld), geology (MineBench), drones (AirSpatial-Bench), maps and travel (MapEval-API).
- **Scale.** From 42 tasks (ShapefileGPT) and 57 (GeoAgentBench) to 13,424 requests (RescueADI) and 100,000 queries (GeoLLM-Engine). A small set gives a wide confidence interval; a large one costs more per run.
- **Training split.** EO-Gym (7.64K), OpenEarthAgent (800), RescueADI (12,426) and VagueEO (5,000) ship training data, so a model may be tested on a distribution it was trained on. UnivEARTH is evaluation-only.
- **Can you run it today?** 13 of the 33 have no public data. TerraBench needs the authors' approval, DORA's release is planned, and AgroTools ships its questions without the tools or the grader.

## Part 3. The evaluation framework, and a real Earth-Bench run

Parts 1 and 2 said what to measure and with which benchmarks. This part is how: the `agentic_evals` harness, then a real Earth-Bench run from end to end.

### 3.1 The harness: one evaluation is one Inspect task

**`agentic_evals` is our evaluation harness, built on top of [Inspect AI](https://inspect.aisi.org.uk/).** Within the EVE project we compare several models, agent designs and datasets, each with its own repository, run command and output format. The harness gives them one interface, in the spirit of [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness): it runs an agent on a public EO benchmark, records **both** the final answer and the tool-call trajectory, and scores the run with **that benchmark's own published metrics**.

A run goes top to bottom through the figure:

- **Run configuration.** A YAML file in `configs/`: model, endpoint, task arguments, generation settings, which samples, how many epochs. An epoch repeats the whole task, which is how Pass@k is measured (section 1.2).
- **Dataset.** The questions, the gold answers and, for trajectory benchmarks, the gold tool calls.
- **Setup.** Runs before the agent on each sample. It starts the benchmark's tools as MCP servers and records which model endpoint answers. A tool error is returned to the model instead of crashing the run.
- **Solver.** The agent. `langgraph_react` is the baseline, and `langgraph_agent(graph)` runs any compiled LangGraph graph you write (section 3.2).
- **Scorers.** The metrics from the benchmark's paper, one module per benchmark (`tasks/<benchmark>/metrics.py`), tested against the published results. A judge outside the paper's protocol is an extra scorer with its own result, never merged into the benchmark's score.
- **Log.** An Inspect `.eval` file with every message, tool call, score and the endpoint identity. Browse it with `inspect view` and rescore it offline without calling the agent again. Weights & Biases and Langfuse can receive the same run.

![agentic_evals, our evaluation harness built on Inspect AI: the run configuration feeds one Inspect task (dataset, setup, solver, scorers), which writes an Inspect log and optionally Weights & Biases and Langfuse](figures/inspect_task.svg)

| Benchmark | Task name | Scored with | Status |
| --- | --- | --- | --- |
| Earth-Bench | `agentic_evals/earth_bench` | Accuracy; TAO, TIO, TEM, ParamAcc, efficiency | ready |
| GeoBenchX | `agentic_evals/geobenchx` | Judge match shares (no match, partial, match) and length difference; step metrics as our labelled addition | ready |
| EO-Gym | `agentic_evals/eo_gym` | Pass@k, judge-based correctness, binary tool metrics, parameters score | port in progress |
| OpenEarthAgent | not yet | not yet | in progress  |

Let us build the Earth-Bench task object and look inside:

In [8]:
from inspect_ai._util.registry import registry_info
from agentic_evals.tasks.earth_bench.task import earth_bench

task = earth_bench(regime="ap", modality="spectrum")
print("samples :", len(task.dataset), "(AP, Spectrum)")
print("setup   :", [s.__qualname__.split(".")[0] for s in task.setup])
print("solver  :", registry_info(task.solver).name, "(default; we pass our own below)")
print("scorers :", [registry_info(s).name for s in task.scorer])
print("metrics the paper reports:", task.metadata["published_metrics"])

samples : 100 (AP, Spectrum)
setup   : ['record_run_identity', 'skip_unavailable_rgb', 'publish_mcp_servers']
solver  : agentic_evals/langgraph_react (default; we pass our own below)
scorers : ['agentic_evals/earth_bench_scorer']
metrics the paper reports: ['accuracy', 'efficiency', 'tao', 'tio', 'tem', 'param_acc']


### 3.2 Solvers

In Inspect, a **solver** is the step that runs on each sample. It receives the sample's messages (the instructions and the question) and returns the finished conversation, with the answer and the trace.

The solver we use is **`langgraph_react`**, a ReAct agent built with LangGraph. It alternates reasoning with tool calls until it gives an answer. The graph behind it is defined in section 3.5.

Any compiled LangGraph graph whose state has a `messages` key can take its place through `langgraph_agent(graph)`. Section 3.6 runs one.

### 3.3 Running it

A run is described in a YAML file in `configs/`, so it can be repeated exactly. The baseline you will run in section 3.7 has this shape:

```yaml
task:
  task: agentic_evals/earth_bench
  args: {regime: ap, modality: spectrum}
model:
  model: openai-api/eve/jsc/alias-eve
  args: {strict_tools: false}
generate_config:
  temperature: 0.1
  max_tokens: 4096
  max_retries: 6
  max_connections: 2
eval_config:
  sample_id: [11, 2, 7, 8, 19, 21, 26, 31, 36, 42]
  max_samples: 4
```

The file `configs/earth_bench/workshop_eve.yaml` in the repository has these settings, with comments.

From a terminal at the repository root:

```bash
uv run inspect eval --run-config configs/earth_bench/workshop_eve.yaml   # run
uv run inspect view                                                     # browse transcripts, traces, scores
uv run inspect score logs/<run>.eval --scorer agentic_evals/earth_bench_scorer   # rescore offline
```

In this notebook we call the same thing from Python (`inspect_ai.eval`), because the agent graph is defined here. `workshop.run()` is a thin wrapper that fixes the task, the question set, and the timeouts.

### 3.4 A subset eval of Earth-Bench

We run a subset eval: ten Spectrum questions in the AP regime, each with a gold tool trajectory. The questions are 11, 2, 7, 8, 19, 21, 26, 31, 36 and 42. Nine need three tool calls, and question 8 needs four. Each question is multiple choice.

#### Two regimes: AP and IF

Earth-Bench asks each question in two regimes. The question is the same; the instructions differ:

- **AP, auto-planning.** The question gives the goal and the choices. The agent plans its own steps, so the score also depends on whether it chose the right tools. We use AP for the headline numbers.
- **IF, instruction-following.** The question lists the steps in order. The agent only has to carry them out, so the score measures whether it follows instructions, not whether it can plan.

Report AP and IF separately. A model can follow listed steps well and still plan poorly, and the two numbers say different things. The cards below show question 11 in both regimes:

In [9]:
import html
import re

from IPython.display import HTML, display

from agentic_evals.tasks.earth_bench.dataset import load_samples

AP = {s.id: s for s in load_samples("ap", "spectrum") if s.id == 11}
IF = {s.id: s for s in load_samples("if", "spectrum") if s.id == 11}


def question_card(title, text, colour):
    parts = re.split(r"\n(?=A\.)", text.strip(), maxsplit=1)
    stem, options = parts[0], (parts[1] if len(parts) > 1 else "")
    stem = stem.replace("benchmark/data/question11", "").strip()
    return (
        f"<div style='flex:1 1 360px;border-left:4px solid {colour};background:rgba(128,128,128,.10);"
        "border-radius:8px;padding:10px 14px;min-width:0'>"
        f"<div style='font-weight:700;margin-bottom:6px'>{title}</div>"
        f"<div style='font-size:13px;line-height:1.5'>{html.escape(stem)}</div>"
        f"<div style='font-size:12px;opacity:.75;margin-top:8px;white-space:pre-line'>{html.escape(options.strip())}</div></div>"
    )


ap_text = AP[11].input.split("Question:", 1)[1]
if_text = IF[11].input.split("Question:", 1)[1]
display(HTML(
    "<div style='display:flex;flex-wrap:wrap;gap:10px'>"
    + question_card("AP: the goal, the agent plans", ap_text, "#0969da")
    + question_card("IF: the same question, the steps listed", if_text, "#8250df")
    + "</div>"
))

In [10]:
import html
import re

from IPython.display import HTML, display

from agentic_evals.tasks.earth_bench.dataset import load_samples

ap = {s.id: s for s in load_samples("ap", "spectrum") if s.id in WORKSHOP_QUESTIONS}
sample = ap[11]

system_prompt, question = sample.input.split("Question:", 1)
parts = re.split(r"\nA\.", question.strip(), maxsplit=1)
stem = parts[0].replace("benchmark/data/question11", "").strip()
options = re.findall(r"^([A-D])\.(.+)$", "A." + parts[1] if len(parts) > 1 else "", re.M)
gold = sample.target
steps = sample.metadata["gold_tool_calls"]

card = "background:rgba(128,128,128,.10);border-radius:8px;padding:10px 14px;margin-bottom:10px"
option_html = "".join(
    f"<div style='padding:2px 0;{'color:#2ea043;font-weight:700' if letter == gold else ''}'>{letter}. {html.escape(text.strip())}</div>"
    for letter, text in options
)
step_html = "".join(
    "<div style='display:flex;gap:10px;align-items:flex-start;margin-bottom:6px'>"
    f"<div style='min-width:22px;height:22px;border-radius:11px;background:#0969da;color:#fff;font-weight:700;"
    f"display:flex;align-items:center;justify-content:center;font-size:12px'>{k}</div>"
    f"<div style='font-family:ui-monospace,Menlo,monospace;font-size:12.5px;overflow-wrap:anywhere'>"
    f"<b>{html.escape(c['name'])}</b>({html.escape(', '.join(f'{a}={v!r}' for a, v in c['input'].items()))})</div></div>"
    for k, c in enumerate(steps, 1)
)
display(HTML(
    "<div style='font-size:13px;line-height:1.5'>"
    f"<div style='{card}'><div style='font-weight:700;margin-bottom:4px'>System prompt (sent as part of the user message)</div>"
    f"<div style='white-space:pre-line'>{html.escape(system_prompt.strip())}</div></div>"
    f"<div style='{card}'><div style='font-weight:700;margin-bottom:4px'>Question (AP), with the data folder and the choices</div>"
    f"<div style='margin-bottom:6px'>{html.escape(stem)}</div>"
    f"<div style='font-size:12px;opacity:.75;margin-bottom:4px'>Data folder: benchmark/data/question11</div>{option_html}</div>"
    f"<div style='{card}'><b>Gold answer:</b> <span style='color:#2ea043;font-weight:700'>{html.escape(gold)}</span></div>"
    f"<div style='{card}'><div style='font-weight:700;margin-bottom:6px'>Gold trajectory</div>{step_html}</div>"
    "</div>"
))

#### Data

The rasters come from the public Hugging Face dataset `Sssunset/Earth-Bench` (the full benchmark is 16.6 GB). We download only the folders for our 10 questions, into this folder's `data/`.

In [11]:
raster_root = download_earth_bench(WORKSHOP_QUESTIONS)
files = [p for p in raster_root.rglob("*") if p.is_file() and ".cache" not in p.parts]
print(f"{len(files)} files in {raster_root}  ({sum(p.stat().st_size for p in files) / 1e6:.1f} MB)")
print(sorted(p.name for p in raster_root.iterdir() if not p.name.startswith(".")))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 590 files:   0%|          | 0/590 [00:00<?, ?it/s]

Rasters for 10 tasks in /Users/antoniolopez/pischool/agentic-evals/evaluating-eo-agents-bifold/data/earth_bench/benchmark/data
590 files in /Users/antoniolopez/pischool/agentic-evals/evaluating-eo-agents-bifold/data/earth_bench/benchmark/data  (1490.5 MB)
['question11', 'question19', 'question2', 'question21', 'question26', 'question31', 'question36', 'question42', 'question7', 'question8']


#### Tools

The agent does not call Python functions directly. Earth-Bench's tools run as five [MCP](https://modelcontextprotocol.io) servers (stdio), one per kit. Any MCP client can use them.

In [12]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from agentic_evals.tasks.earth_bench.tools import KITS, kit_servers

# Each kit starts its own server process. Its diagnostics go to runs/demo/kit_stderr.log.
client = MultiServerMCPClient(kit_servers(Path("runs/demo").resolve()))
for kit in KITS:
    kit_tools = await client.get_tools(server_name=kit)
    print(f"{kit:<11} {len(kit_tools):>3} tools   e.g. {', '.join(t.name for t in kit_tools[:4])}")

Analysis     10 tools   e.g. compute_linear_trend, mann_kendall_test, sens_slope, stl_decompose
Index        12 tools   e.g. calculate_batch_ndvi, calculate_batch_ndwi, calculate_batch_ndbi, calculate_batch_evi
Inversion    17 tools   e.g. band_ratio, lst_single_channel, lst_multi_channel, split_window
Perception   15 tools   e.g. threshold_segmentation, bbox_expansion, count_above_threshold, count_skeleton_contours
Statistics   50 tools   e.g. coefficient_of_variation, skewness, kurtosis, calc_batch_image_mean


In [13]:
# Call one tool by hand, the way the agent will.
tools = await client.get_tools()
by_name = {t.name: t for t in tools}
print(len(tools), "tools in total")
print("Get file list tool out:")
print(await by_name["get_filelist"].ainvoke({"dir_path": "benchmark/data/question11"}))

104 tools in total
Get file list tool out:
[{'type': 'text', 'text': '["2022_08_10_0110_BT_31.tif","2022_08_10_0110_BT_32.tif"]', 'id': 'lc_1f58e46c-8c01-420d-b3b8-9240b66795cc'}]


Every model call carries all these tool descriptions, which is a lot of tokens. Remember this when you design your own graph: a longer loop costs more.

### 3.5 The agent: ReAct

ReAct ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) alternates *reasoning*, *acting* (a tool call) and *observing* (the tool result) until the model answers.

The graph is defined in the next cell. `create_agent` is LangChain's ReAct graph.

`langgraph_agent(react_graph)` is the harness around the graph. It starts the MCP servers, routes `llm` to your model, gives tool errors back to the model instead of crashing, and records every tool call as a trace. The scorer reads that trace. You do not edit the harness.

In [14]:
from langchain.agents import create_agent
from agentic_evals.agents.langgraph_react import NormalizeMistralToolCalls, ToolImagesAsUserMessage

def react_graph(llm, tools):
    return create_agent(
        llm,
        tools=tools,
        middleware=[NormalizeMistralToolCalls(), ToolImagesAsUserMessage()],
        name="workshop_react",
    )

### 3.6 One question, with the raw Inspect call

`workshop.run()`, used later, is a thin wrapper around this call. `strict_tools=False` is required: some Earth-Bench tool schemas have untyped arguments that strict mode rejects.

In [15]:
from inspect_ai import eval as inspect_eval
from workshop import check_budget

check_budget()   # one-token request: stops here if the model's token budget is spent

one = inspect_eval(
    earth_bench(regime="ap", modality="spectrum", solver=langgraph_agent(react_graph, name="workshop_react")),
    model=MODEL,
    model_args={"strict_tools": False},
    sample_id=[11],
    log_dir="logs/one",
    display="none",
    max_tokens=4096,          # cap each reply
    attempt_timeout=180,      # give up on a silent call after 3 minutes
    stream_idle_timeout=60,
    max_retries=6,            # back off and retry on HTTP 429 (the room shares one endpoint)
    max_connections=2,        # at most 2 model calls in flight from this notebook
)[0]
print("status:", one.status)

Output()

status: success


In [16]:
from trace_view import show_trace

show_trace(one, 11)

This is the **execution trace**: every tool call with its arguments and output, in order. Everything that follows (trajectory metrics, the judge, your diagnosis) is computed from it.

#### The score of that sample

In [17]:
from IPython.display import HTML, display
from workshop import compare_html, sample_calls

sample = one.samples[0]
display(HTML(compare_html(sample_calls(sample), ap[11].metadata["gold_tool_calls"], "Question 11: expert and agent, step by step")))

`tool_errors` counts calls that raised an error or returned one (for example a made-up file path). `perception_failures` should stay at 0 here; if it does not, the setup is broken, not the model.

### 3.7 Run a subset

Let's now run the benchmark on 10 samples.

In [ ]:
from IPython.display import HTML, display
from workshop import table_html

baseline = run(
    langgraph_agent(react_graph, name="workshop_react"),
    model=MODEL,
    questions=WORKSHOP_QUESTIONS,
    log_dir="logs/baseline",
    max_samples=10,
)
df_base = results_df(baseline)
display(HTML(table_html(df_base, ["accuracy", "tao", "tio", "tem", "param_acc"], "Per question")))

Output()

Question,accuracy,tao,tio,tem,param_acc
2,0.00,0.67,0.33,0.33,0.33
7,0.00,0.33,0.06,0.06,0.06
8,0.00,0.50,0.25,0.00,0.00
11,1.00,1.00,1.00,0.00,0.00
19,0.00,0.33,0.00,0.00,0.00
21,1.00,1.00,1.00,0.00,0.00
26,0.00,0.33,0.00,0.00,0.00
31,0.00,0.33,0.00,0.00,0.00
36,0.00,0.33,0.00,0.00,0.00
42,0.00,0.33,0.00,0.00,0.00


In [19]:
from IPython.display import HTML, display
from workshop import scores_side_by_side, table_html

COLS = ["accuracy", "tao", "tio", "tem", "param_acc", "efficiency", "tool_errors"]
display(HTML(scores_side_by_side({"Baseline mean": df_base[COLS].mean().to_dict()}, f"Mean over {len(df_base)} questions (Earth-Bench AP, Spectrum)")))

,Baseline mean
Accuracy,0.20
Tool-Any-Order,0.52
Tool-In-Order,0.26
Tool-Exact-Match,0.04
Parameter accuracy,0.04
Efficiency,0.83 calls / expert
Tool errors,2.4


In [20]:
display(HTML(table_html(df_base, ["accuracy", "tio", "tem", "param_acc"], "Accuracy against the trajectory metrics")))

Question,accuracy,tio,tem,param_acc
2,0.00,0.33,0.33,0.33
7,0.00,0.06,0.06,0.06
8,0.00,0.25,0.00,0.00
11,1.00,1.00,0.00,0.00
19,0.00,0.00,0.00,0.00
21,1.00,1.00,0.00,0.00
26,0.00,0.00,0.00,0.00
31,0.00,0.00,0.00,0.00
36,0.00,0.00,0.00,0.00
42,0.00,0.00,0.00,0.00


### 3.8 Bonus: an orchestrator and five specialists, talking over A2A

Every model call of the baseline carries the descriptions of all 104 tools. A multi-agent design splits that load: an **orchestrator** plans the task and hands each step to a **specialist** that owns one group of tools. Each agent sees only a few tools.

The specialists are separate agents, so they need a protocol to talk to each other. We use [A2A (Agent2Agent)](https://a2a-protocol.org), an open protocol between agents: each agent is an HTTP server, and other agents send it JSON-RPC requests. **MCP connects an agent to its tools; A2A connects an agent to other agents.**

These are the A2A objects you will see below (protocol version 1.0):

| A2A object | What it is | In this system |
| --- | --- | --- |
| **AgentCard** | JSON served at `/.well-known/agent-card.json`: name, description, skills with example requests, endpoint URL | one per specialist, written once and committed |
| **Message** | one turn, made of parts (here, text) | the orchestrator's request, or a specialist's question |
| **Task** | the unit of work a message starts. Its state goes from `submitted` to `working`, then to `completed`, `failed` or `input-required` | one per `ask_*` call |
| **Artifact** | the output of a task | the specialist's answer |
| **contextId**, **taskId** | ids that link messages into one conversation, and into one task | a follow-up to the same specialist keeps its history |

#### The specialists

Earth-Bench already divides its 104 tools by MCP kit: one server file per group of related tools. We follow that division, so there is one specialist per kit (Analysis, Index, Inversion, Perception, Statistics), and nothing is clustered. The five specialists are committed in `agents/specialists/clusters/earth_bench.json`, and this notebook loads that file by name: each specialist's card, skills and role prompt (written once by a describer model, so this cell calls no model), and the tools it owns. `get_filelist` stays with the orchestrator, so it can list a folder before it asks a specialist about a file in it.

That file is written by `agents/specialists/clustering.py` (`--task earth_bench`). The same script clusters the tools of a benchmark that has no such division.

In [21]:
import html
import json

from IPython.display import HTML, display

from agentic_evals.agents.specialists.clustering import CLUSTERS_DIR, plan_specialists
from agentic_evals.agents.specialists.graph import without_tools

ORCHESTRATOR_TOOLS = ("get_filelist",)
EARTH_BENCH_PLAN = CLUSTERS_DIR / "earth_bench.json"   # the 5 MCP-kit specialists, committed
specialists = without_tools(await plan_specialists(tools, clusters=str(EARTH_BENCH_PLAN)), ORCHESTRATOR_TOOLS)   # `tools`: the 104 MCP tools of section 3.4

box = "background:rgba(128,128,128,.08);border-radius:8px;padding:10px 12px"
cards = "".join(
    f"<div style='flex:1 1 170px;{box};border-top:4px solid #d97706;min-width:0'>"
    f"<div style='font-weight:700'>{html.escape(s.name)}</div>"
    f"<div style='font-family:ui-monospace,Menlo,monospace;font-size:11px;opacity:.7'>ask_{s.slug}</div>"
    f"<div style='font-size:12px;margin:6px 0'>{len(s.tools)} MCP tools</div>"
    + "".join(f"<div style='font-size:11.5px;opacity:.85'>&bull; {html.escape(k['name'])}</div>" for k in s.skills)
    + "</div>"
    for s in specialists
)
display(HTML(
    "<div style='font-size:13px'>"
    f"<div style='{box};border-top:4px solid #7c3aed;max-width:520px;margin:0 auto'>"
    "<div style='font-weight:700'>Orchestrator: a ReAct agent with no domain tools</div>"
    f"<div style='font-size:12px;opacity:.8'>tools: {len(specialists)} &times; <code>ask_&lt;specialist&gt;</code> + "
    f"{', '.join(ORCHESTRATOR_TOOLS)}. It writes the final answer.</div></div>"
    "<div style='text-align:center;font-size:12px;opacity:.7;margin:6px 0'>&darr; A2A: <code>SendMessage</code>, JSON-RPC over HTTP &darr;"
    " &nbsp; &uarr; a Task: completed with an artifact, input-required, or failed &uarr;</div>"
    f"<div style='display:flex;flex-wrap:wrap;gap:8px'>{cards}</div>"
    "<div style='text-align:center;font-size:12px;opacity:.7;margin:6px 0'>&darr; each specialist is a ReAct agent over its own MCP tools &darr;</div>"
    "<div style='text-align:center;font-size:12px'>Earth-Bench MCP kits: Analysis, Index, Inversion, Perception, Statistics</div>"
    "</div>"
))

#### Start the specialist servers

Each specialist is the ReAct graph of section 3.5 (`create_agent`), built over **its own tools only**. Its system prompt is the role prompt from the describer plus one rule: if the request leaves out something it needs, reply with one line that starts with `NEEDS_INPUT:`.

Two pieces turn the graph into an A2A agent (`agents/specialists/a2a.py`):

- `SubAgentExecutor` runs the graph on each incoming message and turns the reply into a task: `completed` with the answer as an artifact, `input-required` when the reply starts with `NEEDS_INPUT:`, or `failed` if the run raised an error. The task's `contextId` is the graph's `thread_id`, so the specialist remembers earlier requests from the orchestrator.
- `serve()` starts one uvicorn server per specialist on a free localhost port. Each server has two routes: the AgentCard and the JSON-RPC endpoint.

During an eval, the solver sends every model call to the eval's model through `agent_bridge()`. Outside an eval, as here, the specialists call the EVE endpoint directly.

In [22]:
from contextlib import AsyncExitStack

import httpx
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from agentic_evals.agents.langgraph_react import default_middleware
from agentic_evals.agents.specialists.a2a import SubAgentExecutor, agent_card, langgraph_runner, serve
from agentic_evals.agents.specialists.graph import specialist_prompt
from workshop import eve_credentials

model_name, base_url, api_key = eve_credentials()
llm = ChatOpenAI(model=model_name, base_url=base_url, api_key=api_key, temperature=0.1, max_tokens=4096)

servers = {}
for spec in specialists:
    graph = create_agent(
        llm,
        tools=[by_name[t] for t in spec.tools],          # only this specialist's MCP tools
        system_prompt=specialist_prompt(spec, None),
        middleware=default_middleware(None),
        checkpointer=InMemorySaver(),                    # keeps one history per A2A contextId
        name=spec.slug,
    )
    executor = SubAgentExecutor(langgraph_runner(graph, callbacks=[], recursion_limit=30))
    servers[spec.slug] = (executor, lambda url, spec=spec: agent_card(spec.name, spec.description, list(spec.skills), url))

a2a_stack = AsyncExitStack()
urls = await a2a_stack.enter_async_context(serve(servers))   # {slug: "http://127.0.0.1:<port>"}
http = await a2a_stack.enter_async_context(httpx.AsyncClient(timeout=600))

# Each server is running: fetch its AgentCard over HTTP, as any A2A client would.
rows = []
for slug, url in urls.items():
    response = await http.get(f"{url}/.well-known/agent-card.json")
    card = response.json()
    rows.append(
        f"<tr><td style='padding:4px 10px'><b>{html.escape(card['name'])}</b></td>"
        f"<td style='padding:4px 10px;font-family:ui-monospace,Menlo,monospace'>{url}</td>"
        f"<td style='padding:4px 10px;color:#2ea043;font-weight:700'>{response.status_code} OK</td>"
        f"<td style='padding:4px 10px'>{card['version']}</td>"
        f"<td style='padding:4px 10px'>{len(card['skills'])} skills</td></tr>"
    )
display(HTML(
    "<table style='font-size:13px;border-collapse:collapse'><tr style='text-align:left;border-bottom:1px solid rgba(128,128,128,.4)'>"
    "<th style='padding:4px 10px'>A2A agent</th><th style='padding:4px 10px'>URL</th>"
    "<th style='padding:4px 10px'>GET /.well-known/agent-card.json</th><th style='padding:4px 10px'>card version</th>"
    "<th style='padding:4px 10px'>skills</th></tr>" + "".join(rows) + "</table>"
))

A2A agent,URL,GET /.well-known/agent-card.json,card version,skills
Geospatial Analysis Agent,http://127.0.0.1:58058,200 OK,1.0.0+e6d287b0,3 skills
Raster Index Agent,http://127.0.0.1:58059,200 OK,1.0.0+8e4a0203,3 skills
Thermal Remote Sensing Agent,http://127.0.0.1:58060,200 OK,1.0.0+ce5f1309,3 skills
Remote Sensing Vision Agent,http://127.0.0.1:58061,200 OK,1.0.0+2811503d,3 skills
Raster Analysis Agent,http://127.0.0.1:58062,200 OK,1.0.0+8735875f,3 skills


One card in full. This is all the orchestrator knows about the specialist: no tool list, only what it can do and how to ask.

In [23]:
THERMAL = "thermal_remote_sensing"
card = (await http.get(f"{urls[THERMAL]}/.well-known/agent-card.json")).json()
print(json.dumps(card, indent=2))

{
  "name": "Thermal Remote Sensing Agent",
  "description": "Estimates land surface temperature and related thermal or microwave remote-sensing parameters from local raster inputs. It also computes selected vegetation-conditioned temperature statistics, water turbidity, precipitable water vapor, and sea ice concentration.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:58060/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0+ce5f1309",
  "capabilities": {
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "estimate_land_surface_temperature",
      "name": "Estimate land surface temperature",
      "description": "Estimates LST from thermal bands, brightness temperatures, emissivity, or day/night inputs using the supported methods. A request must include all required local raster paths, the output path, and any metho

#### What the orchestrator's model sees

The orchestrator reads each card over HTTP and turns it into one LangChain tool (`ask_tools` in `a2a.py`). With every model call, the orchestrator's model receives each of these tools as a function schema with three parts:

- **name**: `ask_<specialist>`, for example `ask_thermal_remote_sensing`;
- **description**: the card, starting with the agent's name ("Send a request to Thermal Remote Sensing Agent over A2A"), then its description, then each skill with one example request;
- **parameters**: one string, `request`: "What you need done, in plain words, with every file path and value the step needs."

For the model, delegating a step is an ordinary tool call. The tool sends the A2A message, waits for the task to finish, and returns the reply text as the tool result.

In [24]:
import html
import json

from IPython.display import HTML, display
from langchain_core.utils.function_calling import convert_to_openai_tool

from agentic_evals.agents.specialists.a2a import ask_tools

ask = await ask_tools(urls, http)
orchestrator_tools = [*ask, *(by_name[t] for t in ORCHESTRATOR_TOOLS)]

BOX = "background:rgba(128,128,128,.08);border-radius:8px;padding:10px 14px"
LABEL = "font-size:10.5px;font-weight:700;letter-spacing:.6px;opacity:.6;margin:10px 0 4px"
MONO = "font-family:ui-monospace,Menlo,monospace"


def tool_card(tool):
    """One ask_* tool as the orchestrator's model receives it: the function schema's name, description, parameters."""
    fn = convert_to_openai_tool(tool)["function"]
    intro, _, skill_lines = fn["description"].partition("\nSkills:\n")
    skills = []
    for line in skill_lines.splitlines():
        if line.startswith("- "):
            name, _, text = line[2:].partition(": ")
            skills.append([name, text, ""])
        elif line.strip().startswith("Example request:") and skills:
            skills[-1][2] = line.strip().removeprefix("Example request:").strip().strip('"')
    skill_html = "".join(
        f"<li style='margin:6px 0'><b>{html.escape(n)}</b>: {html.escape(t)}"
        + (f"<div style='{MONO};font-size:11.5px;opacity:.75;margin-top:2px'>e.g. \"{html.escape(e)}\"</div>" if e else "")
        + "</li>"
        for n, t, e in skills
    )
    required = fn["parameters"].get("required", [])
    params = "".join(
        f"<div><code>{html.escape(k)}</code> <span style='opacity:.6'>({v.get('type', '')}"
        f"{', required' if k in required else ''})</span>: {html.escape(v.get('description', ''))}</div>"
        for k, v in fn["parameters"]["properties"].items()
    )
    return (
        f"<div style='{BOX};border-top:4px solid #d97706;max-width:820px'>"
        f"<div style='{LABEL}'>NAME</div><div style='{MONO};font-weight:700'>{html.escape(fn['name'])}</div>"
        f"<div style='{LABEL}'>DESCRIPTION</div><div>{html.escape(intro)}</div>"
        f"<ul style='margin:6px 0 0 18px;padding:0'>{skill_html}</ul>"
        f"<div style='{LABEL}'>PARAMETERS</div>{params}</div>"
    )


def schema_chars(ts):
    return sum(len(json.dumps(convert_to_openai_tool(t))) for t in ts)


rows = [("baseline ReAct agent", tools, "#8b949e"), ("orchestrator", orchestrator_tools, "#7c3aed")]
rows += [(s.name, [by_name[t] for t in s.tools], "#d97706") for s in specialists]
sizes = [(name, len(ts), schema_chars(ts), colour) for name, ts, colour in rows]
top = max(chars for _, _, chars, _ in sizes)
bars = "".join(
    f"<tr><td style='text-align:left;padding:3px 12px 3px 0'>{html.escape(name)}</td>"
    f"<td style='text-align:right;padding:3px 12px'>{n}</td>"
    f"<td style='text-align:right;padding:3px 12px;{MONO}'>{chars:,}</td>"
    f"<td style='width:280px;padding:3px 0'><div style='height:10px;width:{100 * chars / top:.1f}%;"
    f"background:{colour};border-radius:3px'></div></td></tr>"
    for name, n, chars, colour in sizes
)
display(HTML(
    "<div style='font-size:13px'>"
    f"<div style='{LABEL}'>ONE ask_* TOOL, AS THE ORCHESTRATOR'S MODEL RECEIVES IT</div>"
    + tool_card(next(t for t in ask if t.name == f"ask_{THERMAL}"))
    + f"<div style='{LABEL};margin-top:18px'>TOOL SCHEMAS SENT WITH EVERY MODEL CALL</div>"
    "<table style='border-collapse:collapse;font-size:12.5px'>"
    "<tr style='opacity:.6'><th style='text-align:left'>agent</th><th style='text-align:right;padding:0 12px'>tools</th>"
    f"<th style='text-align:right;padding:0 12px'>characters</th><th></th></tr>{bars}</table></div>"
))

agent,tools,characters,
baseline ReAct agent,104,"96,536",
orchestrator,6,"9,979",
Geospatial Analysis Agent,10,"9,304",
Raster Index Agent,12,"11,109",
Thermal Remote Sensing Agent,17,"20,131",
Remote Sensing Vision Agent,15,"13,998",
Raster Analysis Agent,49,"41,635",


#### One A2A exchange, by hand

The cell below does what an `ask_*` tool does, with plain HTTP, so you can see the messages. In A2A 1.0 over JSON-RPC a request is one `POST` with the method `SendMessage`.

1. The first request leaves out the file paths. The specialist's prompt tells it to reply `NEEDS_INPUT: ...`, and the executor turns that reply into a task in state `input-required`.
2. The follow-up carries the same `taskId` and `contextId`, so the same task continues, and the specialist still has the first request in its history.

In [25]:
import uuid


async def send(url, text, task=None):
    message = {"messageId": str(uuid.uuid4()), "role": "ROLE_USER", "parts": [{"text": text}]}
    if task is not None:                       # continue the task that asked for input
        message |= {"taskId": task["id"], "contextId": task["contextId"]}
    body = {"jsonrpc": "2.0", "id": 1, "method": "SendMessage", "params": {"message": message}}
    response = await http.post(url, json=body, headers={"A2A-Version": "1.0"})
    return body, response.json()["result"]["task"]


def exchange_html(title, body, task):
    shown = {k: v for k, v in task.items() if k != "history"}    # the history repeats the messages
    state = task["status"]["state"].removeprefix("TASK_STATE_").lower().replace("_", "-")
    colour = {"completed": "#2ea043", "input-required": "#d29922"}.get(state, "#f85149")
    pre = "margin:0;font-size:11.5px;white-space:pre-wrap;overflow-wrap:anywhere;font-family:ui-monospace,Menlo,monospace"
    panel = "flex:1 1 360px;min-width:0;background:rgba(128,128,128,.08);border-radius:8px;padding:10px 12px"
    return (
        f"<div style='font-weight:700;margin:10px 0 6px'>{html.escape(title)} &rarr; "
        f"<span style='color:{colour}'>{state}</span></div><div style='display:flex;flex-wrap:wrap;gap:10px'>"
        f"<div style='{panel};border-left:4px solid #7c3aed'><div style='font-size:12px;opacity:.7'>request: POST {urls[THERMAL]}/</div>"
        f"<pre style='{pre}'>{html.escape(json.dumps(body, indent=2))}</pre></div>"
        f"<div style='{panel};border-left:4px solid {colour}'><div style='font-size:12px;opacity:.7'>response: result.task (history left out)</div>"
        f"<pre style='{pre}'>{html.escape(json.dumps(shown, indent=2))}</pre></div></div>"
    )


first_body, first = await send(
    urls[THERMAL],
    "Compute the land surface temperature with the multi-channel method from the MODIS band 31 and band 32 "
    "brightness temperature rasters.",
)
second_body, second = await send(
    urls[THERMAL],
    "Band 31: benchmark/data/question11/2022_08_10_0110_BT_31.tif. "
    "Band 32: benchmark/data/question11/2022_08_10_0110_BT_32.tif. "
    "Save the result to question11/lst_0110.tif and return its path.",
    task=first,
)
display(HTML(
    exchange_html("1. Request without the file paths", first_body, first)
    + exchange_html("2. Follow-up with the paths, same taskId and contextId", second_body, second)
))
print("same task:", first["id"] == second["id"], "| same context:", first["contextId"] == second["contextId"])

same task: True | same context: True


#### The orchestrator graph

The orchestrator is the `create_agent` ReAct graph again. It differs from the baseline in two ways:

- **Tools:** the five `ask_*` tools plus `get_filelist`.
- **System prompt:** it explains how to delegate:
  - A specialist does not see the conversation, so every request must carry the paths and values it needs.
  - A reply that starts with `[input-required]` must be answered by sending the same specialist the missing information.

In [26]:
from agentic_evals.agents.specialists.graph import orchestrator_prompt

orchestrator = create_agent(
    llm,
    tools=orchestrator_tools,
    system_prompt=orchestrator_prompt(ORCHESTRATOR_TOOLS),
    middleware=default_middleware(None),
    name="orchestrator",
)
print(orchestrator_prompt(ORCHESTRATOR_TOOLS))
print("\ngraph nodes:", [n for n in orchestrator.get_graph().nodes if not n.startswith("__")])
print("tools:", [t.name for t in orchestrator_tools])

await a2a_stack.aclose()   # stop the demo servers; the solver below starts its own for each sample

# How to work

Specialist agents run the domain tools. Each owns one group of tools, and you reach one over A2A with its `ask_<name>` tool. Each tool's description is that agent's card: what it does and its skills, with an example request. Break the task into steps and give each step to the specialist whose skills fit it. You also have a few tools of your own, listed after this text; use one when a step needs its result, for example to list a directory before you ask about a file in it.

- A specialist does not see the conversation. Put in each request everything the step needs: the file paths, values and earlier results it should use, and what it should return.
- A specialist keeps its own history, so a follow-up to the same one can refer to its earlier answer.
- A reply that starts with [input-required] is a question about your request: the specialist cannot go on without what it names. Answer it with that information, taken from the task or from earlier results (or get it with one o

#### Run the specialists graph

Now we run the setup shown above on question 11, through the harness, with the implementation already in the library: `langgraph_specialists` (`agents/specialists/graph.py`).

In [27]:
from agentic_evals.agents.specialists import langgraph_specialists

multi = run(
    langgraph_specialists(name="a2a_specialists", orchestrator_tools=ORCHESTRATOR_TOOLS),
    model=MODEL,
    questions=[11],
    log_dir="logs/specialists",
    max_samples=1,
)
print("status:", multi.status)

Output()

status: success


How to read this trace:

- **Purple** cards are the orchestrator's model turns. 
- **Amber** cards are A2A tasks: the request the orchestrator wrote, the task state and ids, and the reply it got back.
- **Teal** tool calls.

What to look at:
- **Hand-offs.** A specialist sees only the request the orchestrator wrote. A missing path or a wrong unit in that request is a new kind of failure, and it is visible in the trace as an `input-required` task or as a tool error inside the specialist.
- **Same metrics.** The trajectory metrics read the MCP calls of all agents together, so this system is scored on the same metrics as a single agent.

In [28]:
show_trace(multi, 11)

In [29]:
from inspect_ai.log import list_eval_logs, read_eval_log
from workshop import compare_html, sample_calls, scores_side_by_side

display(HTML(compare_html(sample_calls(multi.samples[0]), ap[11].metadata["gold_tool_calls"], "Question 11: expert and A2A specialists, step by step")))


def cost(log):
    sample = log.samples[0]
    usage = sum((u.total_tokens for u in log.stats.model_usage.values()), 0)
    return sum(1 for e in sample.events if e.event == "model"), usage


single = one if "one" in globals() else read_eval_log(list_eval_logs("logs/one")[0])   # the run of section 3.6
runs = {}
for label, log in (("ReAct, one agent (3.7)", single), ("A2A specialists", multi)):
    row = results_df(log).loc[11].to_dict()
    calls, tokens = cost(log)
    runs[label] = row
    print(f"{label:<24} model calls {calls:>3}   tokens {tokens:>9,}")
display(HTML(scores_side_by_side(runs, "Question 11")))

ReAct, one agent (3.7)   model calls   5   tokens   142,701
A2A specialists          model calls  14   tokens    88,492


,"ReAct, one agent (3.7)",A2A specialists
Accuracy,1.00,1.00
Tool-Any-Order,1.00,1.00
Tool-In-Order,1.00,1.00
Tool-Exact-Match,0.00,0.00
Parameter accuracy,0.00,0.00
Efficiency,1.33 calls / expert,1.67 calls / expert
Tool errors,1,2


## Wrap-up

What you did:

1. Surveyed EO-agent benchmarks and saw that each one scores something different, with its own definitions.
2. Separated answer-only from trajectory evaluation, and computed Accuracy, TAO, TIO, TEM and ParamAcc.
3. Ran an agent through a harness that records every tool call, and read the trace of a failure.
4. Saw how a judge's verdict is read, and why it must be reported with the score.


### Next steps

- Experiment with more multi-agent graphs: split the tools differently, change how the orchestrator plans, or let specialists ask each other, and compare each design with the baseline on the same metrics.
- Explore the A2A protocol beyond: longer `input-required` exchanges, and specialists that run on other machines or are built with other frameworks.
- Integrate more benchmarks into the harness.

### Troubleshooting

- `401` or `403` from the model: check `EVE_API_KEY` and `EVE_BASE_URL` in this folder's `.env`.
- `404 model not found`, or `check_environment` says the endpoint does not list the model: check `MODEL_NAME`. The check is `GET {EVE_BASE_URL}/models`, the same call the harness records on each sample.
- `BudgetExceeded`: your key's token budget is spent; the message says when it resets. Retrying will not help; ask the organizers for a key with budget. A full pass of this notebook uses about 1.8M tokens.
- `RateLimitError` or HTTP 429: the endpoint is busy. `run()` already backs off and retries; if it persists, lower `max_samples` and wait a minute.
- `Wrong kernel` in the setup cell, or `ModuleNotFoundError`: select the kernel **agentic-evals workshop** (top right in Jupyter and VS Code).
- Many `tool_errors`: read the trace with `show_trace`. Usually the model invented a file path.
- A run is very slow: lower `max_samples` or test on `WORKSHOP_QUESTIONS[:3]`.
- Import errors after install: restart the kernel and run the setup cell again.

### References

- Feng et al. *Earth-Agent: Unlocking the Full Landscape of Earth Observation with Agents* (Earth-Bench). ICLR 2026. [arXiv:2509.23141](https://arxiv.org/abs/2509.23141)
- Krechetova & Kochedykov. *GeoBenchX: Benchmarking LLMs in Agent Solving Multistep Geospatial Tasks.* SIGSPATIAL 2025. [arXiv:2503.18129](https://arxiv.org/abs/2503.18129)
- *EO-Gym: A Multimodal, Interactive Environment for Earth Observation Agents.* [arXiv:2605.01250](https://arxiv.org/abs/2605.01250)
- Shabbir et al. *ThinkGeo: Evaluating Tool-Augmented Agents for Remote Sensing Tasks.* [arXiv:2505.23752](https://arxiv.org/abs/2505.23752)
- Kao et al. *Towards LLM Agents for Earth Observation* (UnivEARTH). [arXiv:2504.12110](https://arxiv.org/abs/2504.12110)
- Hou et al. *Can Large Language Models Generate Geospatial Code?* (GeoCode-Bench). [arXiv:2410.09738](https://arxiv.org/abs/2410.09738)
- Dihan et al. *MapEval: A Map-Based Evaluation of Geo-Spatial Reasoning in Foundation Models.* ICML 2025. [arXiv:2501.00316](https://arxiv.org/abs/2501.00316)
- Yao et al. *ReAct: Synergizing Reasoning and Acting in Language Models.* [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
- Chen et al. *Evaluating Large Language Models Trained on Code* (Pass@k: §2.1, Eq. 1; unbiasedness in Appendix A). [arXiv:2107.03374](https://arxiv.org/abs/2107.03374)
- Kulal et al. *SPoC: Search-based Pseudocode to Code.* NeurIPS 2019. [arXiv:1906.04908](https://arxiv.org/abs/1906.04908)
- OpenAI. *Spinning Up in Deep RL: Key Concepts in RL* (trajectory probability). [spinningup.openai.com](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html)
- Atrio et al. *EVE: A Domain-Specific LLM Framework for Earth Intelligence.* ACL 2026.
- [Inspect AI](https://inspect.aisi.org.uk/), [LangGraph](https://github.com/langchain-ai/langgraph), [Model Context Protocol](https://modelcontextprotocol.io), [A2A (Agent2Agent) protocol](https://a2a-protocol.org), [agentevals](https://github.com/langchain-ai/agentevals)